# Example Usage

This notebook demonstrates the core capabilities of the our model, including:
- Text-input Conversation
- Speech-input Conversation
- Automatic Speech Recognition (ASR)
- Audio Captioning
- Text-to-Speech (TTS)

Our model is a novel AudioLLM that supports unified audio understanding and interleaved text and speech generation.

## Environment Setup and Model Loading

First, import the necessary libraries and load the model:

- `model_path`: Path to the main model weights
- `audio_decoder_path`: Path to the audio decoder from [StableToken](https://github.com/Tencent/StableToken).
- `device`: Inference device (GPU recommended)

`generation_config` contains generation parameters. You can adjust `temperature`, `top_p`, and other sampling strategies as needed.

In [1]:
import os
import warnings
warnings.filterwarnings('ignore')
import torchaudio
from src.model import UASAudio

model = UASAudio(
    model_path="checkpoints/Unified_Audio_Schema",
    audio_decoder_path="checkpoints/StableToken/decoder",
    device="cuda"
)
os.makedirs("outputs", exist_ok=True)

dialogue_system_prompt = "User will provide you with a speech instruction. Do it step by step. First, think about \
the instruction and respond in a interleaved manner, with 13 text token followed by 52 audio tokens."
asr_prompt = "Transcribe the following audio content into text."
caption_prompt = "Describe the content of the following audio."
tts_prompt = "Convert the following text into audio"

generation_config = {
    "max_new_tokens": 4096,
    "temperature": 0.7,
    "repetition_penalty": 1.05,
    "top_p": 0.9,
    "do_sample": True
}

def generate_response(model, messages, generation_config=generation_config, speech_output_path="outputs/response.wav"):
    tokens, text, audio_tokens = model(messages, **generation_config)
    if text:
        print(f"\nText response: {text}")
    else:
        print(f"\nNo text response generated.")
    if len(audio_tokens) > 0:
        audio_array, sampling_rate = model.tokens_to_audio(audio_tokens)
        speech_length = audio_array.shape[1] / sampling_rate
        torchaudio.save(speech_output_path, audio_array, sampling_rate)
        print(f"\nSpeech response saved to `{speech_output_path}`, length: {speech_length:.2f} seconds.")
    else:
        print(f"\nNo speech response generated.")

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

## Example 1: Text-input Conversation

In this example, the user provides input as **plain text**, and the model generates an **interleaved text + speech** response.

- **Input**: Text instruction "Give me a brief introduction to the Great Wall."
- **Output**: Model-generated text response with corresponding speech audio

In [2]:
messages = [
    {"role": "system", "content": dialogue_system_prompt},
    {"role": "user", "content": "Give me a brief introduction to the Great Wall."},
    {"role": "assistant", "content": None}
]
output_speech_path = "outputs/example_1_response.wav"
generate_response(model, messages, generation_config, output_speech_path)


Text response: The Great Wall of China is an ancient series of fortifications built along the northern borders of China to protect against invasions and raids by nomadic groups from the north. It is one of the most iconic symbols of Chinese civilization and engineering prowess. Construction of the wall began as early as the seventh century BC, with the most well known sections built by the Ming Dynasty between thirteen sixty eight and sixteen forty four. The wall stretches over thirteen thousand miles, making it the longest wall in the world. It is made of various materials including stone, brick, and earth, depending on the region and available resources. The Great Wall is not a single continuous structure but rather a series of walls and fortifications built by different dynasties over centuries. Today, it is a UNESCO World Heritage Site and attracts millions of visitors each year who come to witness its grandeur and historical significance.



Speech response saved to `outputs/example_1_response.wav`, length: 57.43 seconds.


## Example 2: Speech-input Conversation

This example demonstrates the complete **speech-in → speech-out** dialogue flow.

- **Input**: User's speech audio file (asking about the Great Wall)
- **Output**: After understanding the speech content, the model generates an interleaved text + speech response

In [3]:
messages = [
    {"role": "system", "content": dialogue_system_prompt},
    {"role": "user", "content": [
        {"type": "audio", "audio": "assets/give_me_a_brief_introduction_to_the_great_wall.wav"}
    ]},
    {"role": "assistant", "content": None}
]
output_speech_path = "outputs/example_2_response.wav"
generate_response(model, messages, generation_config, output_speech_path)


Text response: The Great Wall of China is one of the most iconic structures in the world. It was originally built by different Chinese dynasties, with the most famous sections constructed during the Ming Dynasty between thirteen sixty-eight and sixteen forty-four. The wall stretches over thirteen thousand miles, making it the longest man-made structure on Earth. It was built to protect Chinese states and empires from invasions and raids by nomadic groups from the north. The Great Wall also serves as a symbol of Chinese civilization and determination. Today, it is a UNESCO World Heritage Site and one of the most visited tourist attractions in China.



Speech response saved to `outputs/example_2_response.wav`, length: 38.51 seconds.


## Example 3: Automatic Speech Recognition (ASR)

This example demonstrates the model's **speech-to-text** capability.

- **Input**: Speech audio file
- **Output**: Transcribed text content

In [4]:
messages = [
    {"role": "user", "content": [
        {"type": "text", "text": asr_prompt},
        {"type": "audio", "audio": "assets/give_me_a_brief_introduction_to_the_great_wall.wav"}
    ]},
    {"role": "assistant", "content": None}
]
output_speech_path = "outputs/example_3_response.wav"
generate_response(model, messages, generation_config, output_speech_path)


Text response: give me a brief introduction to the great wall

No speech response generated.


## Example 4: Audio Captioning

This example showcases the model's **audio understanding** capability for describing the overall content of audio.

- **Input**: Any audio file
- **Output**: Text description of the audio content

In [5]:
messages = [
    {"role": "user", "content": [
        {"type": "text", "text": caption_prompt},
        {"type": "audio", "audio": "assets/give_me_a_brief_introduction_to_the_great_wall.wav"}
    ]},
    {"role": "assistant", "content": None}
]
output_speech_path = "outputs/example_4_response.wav"
generate_response(model, messages, generation_config, output_speech_path)


Text response: The audio clip features a single, professionally recorded male voice delivering the phrase, “Give me a brief introduction to the Great Wall,” in clear, neutral American English. The setting is acoustically controlled and free of background noise, with only a faint electronic hiss present. The speaker’s tone is calm and formal, suggesting an educational or documentary context. After the spoken phrase, a synthesized musical chord is introduced, characterized by a low-frequency, sustained, and harmonically rich sound with a buzzy, sawtooth-like timbre. The chord is static, dry, and unmodulated, serving as a non-diegetic transition or underscore that evokes a sense of gravity and anticipation. The overall production is high-fidelity, with no distortion or artifacts, and the structure and style indicate its use in a modern educational or documentary media format.

No speech response generated.


## Example 5: Text-to-Speech (TTS)

This example demonstrates the model's **speech synthesis** capability, converting plain text into natural speech.

- **Input**: Text content to be spoken
- **Output**: Synthesized speech audio

In [6]:
messages = [
    {"role": "user", "content": [
        {"type": "text", "text": f"{tts_prompt}: Give me a brief introduction to the Great Wall."}
    ]},
    {"role": "assistant", "content": None}
]
output_speech_path = "outputs/example_5_response.wav"
generate_response(model, messages, generation_config, output_speech_path)


No text response generated.



Speech response saved to `outputs/example_5_response.wav`, length: 3.52 seconds.
